# Test et Évaluation du Modèle YOLOv8-seg (Faisceaux de câbles)

Ce notebook permet de tester et d'évaluer le modèle entraîné (`best_ep100.pt`) sur :
1. **Le jeu de test complet (`test`)** : calcul des métriques globales et par classe (`Precision`, `Recall`, `Box mAP50`, `Mask mAP50`, `Mask mAP50-95`).
2. **L'inférence visuelle sur les images de test** : superposition des masques pixel par pixel, contours exacts, boîtes englobantes et scores de confiance.
3. **L'exportation des résultats structurés** (JSON / statistiques par image) et le **test sur une image externe**.

## 1. Importation des bibliothèques

In [9]:
import json
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

## 2. Configuration des chemins et chargement du modèle entraîné

In [10]:
DATASET_DIR = Path(r"../data/processed/wiring harness connectors segmen.v1i.yolov8").resolve()
DATA_YAML = DATASET_DIR / "data.yaml"
TEST_IMAGES_DIR = DATASET_DIR / "test" / "images"

MODELS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models").resolve()
CHECKPOINTS_DIR = MODELS_DIR / "01_detection_et_segmentation_paliers"
MODEL_PATH = CHECKPOINTS_DIR / "best_ep100.pt"

OUTPUT_DIR = Path(r"D:\S5\Projet_d_expertise\projet/tests/01_test_segmentation_yolov8_seg").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Modèle introuvable : {MODEL_PATH}")
if not DATA_YAML.exists():
    raise FileNotFoundError(f"Fichier data.yaml introuvable : {DATA_YAML}")

model = YOLO(str(MODEL_PATH))
print(f"Modèle chargé avec succès : {MODEL_PATH}")
print(f"Classes détectées : {model.names}")

Modèle chargé avec succès : D:\S5\Projet_d_expertise\projet\models\01_detection_et_segmentation_paliers\best_ep100.pt
Classes détectées : {0: 'cable', 1: 'cable-tie', 2: 'clip', 3: 'connector', 4: 'label'}


## 3. Évaluation quantitative sur le split `test` (`model.val(split='test')`)

In [15]:
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.6,
    project=str(OUTPUT_DIR),
    name="eval_split_test",
    exist_ok=True,
    plots=True,
)

print("\n=== RÉSUMÉ DES MÉTRIQUES GLOBALES (SPLIT TEST) ===")
print(f"Box  - Precision : {metrics.box.mp:.4f} | Recall : {metrics.box.mr:.4f} | mAP50 : {metrics.box.map50:.4f} | mAP50-95 : {metrics.box.map:.4f}")
print(f"Mask - Precision : {metrics.seg.mp:.4f} | Recall : {metrics.seg.mr:.4f} | mAP50 : {metrics.seg.map50:.4f} | mAP50-95 : {metrics.seg.map:.4f}")

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
val: Fast image access  (ping: 0.10.0 ms, read: 55.913.8 MB/s, size: 30.5 KB)
val: Scanning D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8\test\labels.cache... 11 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 11/11  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.6s/it 5.1s<10.4s
                   all         11         25      0.546      0.653      0.653      0.537      0.596      0.729      0.708      0.554
                 cable          1          4      0.286        0.5      0.214     0.0875      0.429       0.75      0.375       0.12
             cable-tie          2          2          1        0.5       0.75        0.6          1        0.5       0.75      0.525
             connector          8         18      0.647      0.611   

## 4. Inférence et visualisation des masques pixel par pixel sur les images de test

In [16]:
CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.5

test_images = sorted(
    [p for p in TEST_IMAGES_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}]
)
print(f"Nombre d'images dans le dossier test : {len(test_images)}")

results = model.predict(
    source=str(TEST_IMAGES_DIR),
    imgsz=640,
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    retina_masks=True,
    save=True,
    project=str(OUTPUT_DIR),
    name="predictions_visuelles",
    exist_ok=True,
)

# Affichage d'une grille de résultats (masques + contours exacts)
n_show = min(len(results), 6)
if n_show > 0:
    cols = 2
    rows = (n_show + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = np.array(axes).reshape(-1)

    for idx in range(len(axes)):
        ax = axes[idx]
        if idx < n_show:
            plotted_bgr = results[idx].plot(masks=True, boxes=True, labels=True, conf=True)
            plotted_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)
            ax.imshow(plotted_rgb)
            ax.set_title(f"{Path(results[idx].path).name} ({len(results[idx].boxes)} détections)")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

Nombre d'images dans le dossier test : 11
Results saved to D:\S5\Projet_d_expertise\projet\tests\01_test_segmentation_yolov8_seg\predictions_visuelles


<Figure size 1400x1500 with 6 Axes>

## 5. Extraction détaillée des masques, surfaces (en pixels) et export JSON

In [17]:
detections_report = []
global_class_counts = Counter()

for res in results:
    img_name = Path(res.path).name
    img_h, img_w = res.orig_shape
    instances = []

    if res.boxes is not None and len(res.boxes) > 0:
        boxes_xyxy = res.boxes.xyxy.cpu().numpy()
        classes_id = res.boxes.cls.cpu().numpy().astype(int)
        confs = res.boxes.conf.cpu().numpy()
        masks_data = res.masks.data.cpu().numpy() if res.masks is not None else None
        polygons_xy = res.masks.xy if res.masks is not None else None

        for i in range(len(boxes_xyxy)):
            cls_name = model.names[classes_id[i]]
            global_class_counts[cls_name] += 1

            mask_area_px = int(masks_data[i].sum()) if masks_data is not None else 0
            polygon_pts = polygons_xy[i].round(2).tolist() if polygons_xy is not None else []

            instances.append({
                "instance_id": i + 1,
                "class": cls_name,
                "confidence": round(float(confs[i]), 4),
                "bbox_xyxy": [round(float(v), 2) for v in boxes_xyxy[i]],
                "mask_area_pixels": mask_area_px,
                "polygon_points_count": len(polygon_pts),
            })

    detections_report.append({
        "image": img_name,
        "resolution": [img_w, img_h],
        "total_instances": len(instances),
        "instances": instances,
    })

report_path = OUTPUT_DIR / "test_predictions_summary.json"
report_path.write_text(json.dumps(detections_report, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Rapport JSON sauvegardé : {report_path}")
print(f"Répartition des instances détectées sur le jeu de test : {dict(global_class_counts)}")

Rapport JSON sauvegardé : D:\S5\Projet_d_expertise\projet\tests\01_test_segmentation_yolov8_seg\test_predictions_summary.json
Répartition des instances détectées sur le jeu de test : {'connector': 14, 'label': 4, 'cable': 6, 'clip': 1, 'cable-tie': 1}


## 6. Test sur une nouvelle image individuelle (et affichage des masques isolés)

In [18]:
def test_single_image(image_path: str | Path, conf: float = 0.25):
    """Teste le modèle sur une image donnée et affiche l'image segmentée + les masques binaires."""
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"Image introuvable : {image_path}")

    pred = model.predict(source=str(image_path), imgsz=640, conf=conf, retina_masks=True, verbose=False)[0]
    vis_rgb = cv2.cvtColor(pred.plot(masks=True, boxes=True, labels=True, conf=True), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(vis_rgb)
    axes[0].set_title(f"Segmentation complète : {image_path.name}")
    axes[0].axis("off")

    if pred.masks is not None:
        combined_mask = np.zeros(pred.orig_shape, dtype=np.uint8)
        for idx_m, m in enumerate(pred.masks.data.cpu().numpy(), start=1):
            combined_mask[m > 0.5] = idx_m
        axes[1].imshow(combined_mask, cmap="nipy_spectral")
        axes[1].set_title(f"Masques d'instances pixel par pixel ({len(pred.masks)} objets)")
    else:
        axes[1].text(0.5, 0.5, "Aucune instance détectée", ha="center", va="center")
        axes[1].set_title("Masques d'instances")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

# Exemple d'exécution sur la première image de test disponible
if test_images:
    test_single_image(test_images[0], conf=CONF_THRESHOLD)

<Figure size 1400x600 with 2 Axes>